## Introduction

In this tutorial, we will build a character-level text autocomplete model using a Recurrent Neural Network (RNN) in PyTorch. We will train the model on the text from "warandpeace.txt". This project will help you understand how RNNs can be implemented for text generation tasks and their application in building your own autocomplete model.


## Importing Necessary Libraries

In [1]:
# This is Cell #1

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import random
import re


## Setting Up the Device

In [2]:
# This is Cell #2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## Reading and Preprocessing the Data

Now it is time to prepare our training data.


In [3]:
# This is Cell #3

def read_file(filename):
    with open(filename, "r", encoding="utf-8") as file:
        text = file.read().lower()
        # Keep only lowercase letters and standard punctuation (.,!?;:()[])
        text = re.sub(r'[^a-z.,!?;:()\[\] ]+', '', text)
    return text

sequence = read_file("warandpeace.txt")


### Here we will train our model with a simple sequence

We will start by training our model with a simple sequence and repettitive sequence such as `"abcdefghijklmnopqrstuvwxyzabcdef..."`, and we will see if our RNN is capable of learning that pattern or not. This will help you easily verify if your RNN is working correctly or not.

In [4]:
# This is Cell #4

#sequence = "abcdefghijklmnopqrstuvwxyz" * 100

## Create Character Mappings

Creating character mappings is essential because RNNs require numerical input to process data. By mapping each unique character to an index and creating a reverse mapping, we convert text data into numerical sequences that the model can understand. This step allows us to encode input text for training and decode the model's output back into readable characters during text generation.



In [5]:
# Cell #5

# Create a list of unique characters from the text sequence
vocab = list(set(sequence))  # Get listed unique characters

# Create two dictionaries for character-index mappings that map each character in vocab to a unique index and vice versa
char_to_idx = {char: idx for idx, char in enumerate(vocab)}  # Create a mapping from character to index
idx_to_char = {idx: char for idx, char in enumerate(vocab)}  # Create a mapping from index to character

# Convert the entire text based data into numerical data
data = [char_to_idx[char] for char in sequence]  # Convert each character in the sequence to its corresponding index


## Defining the CharDataset Class

Now we will create a custom dataset class to generate sequences and targets for training

Creating a custom `CharDataset` class is crucial because it prepares our text data into input sequences and target sequences that the RNN can learn from. By organizing the data this way, we can efficiently feed batches of sequences into the model during training, allowing it to learn the patterns of character sequences in the text.

In [6]:
# This is Cell #6

class CharDataset(Dataset):
    def __init__(self, data, sequence_length, stride, vocab_size):
        self.data = data
        self.sequence_length = sequence_length
        self.stride = stride
        self.vocab_size = vocab_size
        self.sequences = []
        self.targets = []
        
        # Create overlapping sequences with stride
        for i in range(0, len(data) - sequence_length, stride):
            self.sequences.append(data[i:i + sequence_length])
            self.targets.append(data[i + 1:i + sequence_length + 1])

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
        target = torch.tensor(self.targets[idx], dtype=torch.long)
        return sequence, target
    

## Setting Hyperparameters

Now we will set our model's hyperparameters for our training process

Setting hyperparameters is important because they define the model's architecture and training behavior. They determine how the RNN processes data, learns patterns, and how quickly it converges during training. Properly chosen hyperparameters can significantly improve model performance and is a key step in training of models

Set the following hyperparameters for your model in the code cell below:
`sequence_length`, `stride`, `embedding_dim`, `hidden_size`, `num_layers`, `learning_rate`, `num_epochs`, `batch_size`, `vocab_size`.

In [7]:
# This is Cell #7

#TODO: Set your model's hyperparameters

sequence_length = 100  # Length of each input sequence
stride = 2            # Stride for creating sequences
embedding_dim = 30     # Dimension of character embeddings
hidden_size = 100      # Number of features in the hidden state of the RNN
learning_rate = 0.007  # Learning rate for the optimizer
num_epochs = 1         # Number of epochs to train
batch_size = 128        # Batch size for training
vocab_size = len(vocab)
input_size = len(vocab)
output_size = len(vocab)


After you have set your hyperparameters in the code cell above, very breifly tell what is the role of each of the hyperparameter that you have defined above.

TODO: Explain below

> After hours of testing and optimizing my hyperparameters (I have a very old laptop), the parameters above did a decent job while also not taking very long to run. Below is my understanding of the trends of each hyperparameter (considering the fact that I used the Adam optimizer).

sequence_length: the larger the sequence the length the more context you'd capture but increases time and storage complexity of the model. 

stride: larger the stride, the more characters you skip over in between stretches. The lower the stride the better the accuracy because you skip over fewer characters, but this increases the complexity by quite a lot (based on the length of the text/document).

embedding_dim: the amount of dimensions it takes to represent the input text. Therefore, you need a higher value for this for more complex text to be understood. 

hidden_size: you typically need more layers if the text is longer and more complex. Therefore, for warandpeace, this value needs to be around the 100s.

learning_rate: Proportional to the number of epochs. For the Adam optimizer, I found that it pretty much had to be lower than 0.1. You need to find a good medium however, because it its too high it will overfit to the data, and if its too low if won't learn the data well. 

num_epochs: The number of runs through the data. Therefore, for long texts you can't have this value be too high without increasing the complexity by quite a bit. You need to find a value for the epochs so that you don't overtrain your model (too high of a value), and don't have it low enough for the model to stop learning too early. 

batch_size: the number of sequences that you send into the modal at once. The larger the batch_size the more memory the model has to hold and process at once. While increasing it does increase space complexity, time complexity decreases.

vocab_size, input_size, output_size: these are just the size of the vocabulary in the text.

It is important to understand that these hyperparameters are set based on your text. There is no "higher the better" or "lower the better" answer, that is why you have to optimize them and that is why there is so much research in the field of optimizing them. You have to find a middle ground for it to not overtrain or undertrain, while also sticking to the restrictions of your hardware. 
> 

## Splitting Data into Training and Testing Sets

By now at this point in class, I'm confident that you know why we do this, so I'm not gonna say a lot here, let's jump right into the todo.

In [8]:
# Cell #8

data_tensor = torch.tensor(data, dtype=torch.long)

# Convert the data into a pytorch tensor and split the data into 90:10 ratio
train_size = int(0.9 * len(data_tensor))  # 90% of the data for training
train_data = data_tensor[:train_size]      # Training data
test_data = data_tensor[train_size:]        # Testing data


## Creating Data Loaders

Now we will create data loaders for easy batching during training and testing.

Creating data loaders is essential to batch the data during training and testing. Batching allows the RNN to process multiple sequences in parallel, which speeds up training and makes better use of computational resources. 
We will also use Data loaders to shuffle the batched data, which is important for training models that generalize well.

Make sure to set `drop_last=True`

In [9]:
# Cell #9

train_dataset = CharDataset(train_data, sequence_length, stride, vocab_size)
test_dataset = CharDataset(test_data, sequence_length, stride, vocab_size)

# Initialize the training and testing data loader with batching and shuffling equal to True for training (and shuffling = False for testing)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)  # Training data loader
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)    # Testing data loader

total_batches = len(train_loader)


## Defining the RNN Model

Here we will define our character-level RNN model.

In [10]:
# This is Cell #10

class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, embedding_dim=30):
        super(CharRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = torch.nn.Embedding(output_size, embedding_dim)
        self.W_e = nn.Parameter(torch.randn(hidden_size, embedding_dim) * 0.01)  # Smaller std
        self.b_e = nn.Parameter(torch.zeros(hidden_size))
        self.W_h = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)  # Smaller std
        self.b_h = nn.Parameter(torch.zeros(hidden_size)) 
        #TODO: set the fully connected layer
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden):
        """
        x in [b, l] # b is batch_size and l is sequence length
        """
        x_embed = self.embedding(x)  # [b=batch_size, l=sequence_length, e=embedding_dim]
        b, l, _ = x_embed.size()
        x_embed = x_embed.transpose(0, 1) # [l, b, e]
        if hidden is None:
            h_t_minus_1 = self.init_hidden(b)
        else:
            h_t_minus_1 = hidden
        output = []
        for t in range(l):
            # RNN equation from the lecture 
            # We add a bias as well to expand the range of learnable functions
            h_t = torch.tanh(x_embed[t] @ self.W_e.T + self.b_e + h_t_minus_1 @ self.W_h.T + self.b_h) # [b, e]
            output.append(h_t)
            h_t_minus_1 = h_t
        output = torch.stack(output) # [l, b, e]
        output = output.transpose(0, 1) # [b, l, e]
        final_hidden = h_t.clone() # [b, h]
        logits = self.fc(output) # [b, l, vocab_size=v] 
        return logits, final_hidden
    
    def init_hidden(self, batch_size):
        return torch.zeros(batch_size, self.hidden_size).to(device)


For a basic high level understanding of what is the CharRNN model that you just defined above, it consists of an embedding layer, an RNN layer, and a fully connected layer. Then embedding layer converts character indices into embeddings. Then RNN processes the embeddings and captures sequential information. Then finally the fully connected layer maps the RNN outputs to the vocabulary size for character prediction.


# Initializing the Model, Loss Function, and Optimizer

Now we will create an instance of the model that we just defined above and set up the loss function and optimizer. Then we will define a loss function, that evaluates the model's prediction against the true targets, and attaches a cost (number) on how good/bad the model is doing. During our training process, it is this cost that we try to minimize by tweaking the weights of the network. 

Then we will set up an optimizer, which will update the model's parameters based on the loss returned by the our loss function. This is how our model will learn over time.


In [11]:
# Cell #12

# Initialize your RNN model
model = CharRNN(input_size=vocab_size, hidden_size=hidden_size, output_size=vocab_size, embedding_dim=embedding_dim).to(device)

# Define the loss function (use cross entropy loss)
criterion = nn.CrossEntropyLoss()

# Initialize your optimizer passing your model parameters and training hyperparameters
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


## Training the Model

Now finally, after all the setup that we have done, we can train our RNN. 

A basic idea high level idea of what we will do here is we will loop over epochs and batches to train the model. 
We will Initialize the hidden state at the beginning of each epoch. For each batch, we will reset the gradients, perform a forward pass, compute the loss, perform backpropagation, and update the model parameters. Then we detach the hidden state to prevent gradients from backpropagating through previous batches. We ill repeat this process for each batch. And finally we will calculate the average loss and accuracy for each epoch.
By performing forward and backward passes, calculating loss, and updating the model parameters, we enable the RNN to improve its predictions with each epoch.

In [12]:
# This is Cell #13

for epoch in range(num_epochs):
    total_loss, correct_predictions, total_predictions = 0, 0, 0

    hidden = model.init_hidden(batch_size)

    for batch_idx, (batch_inputs, batch_targets) in tqdm(enumerate(train_loader), total=total_batches, desc=f"Epoch {epoch+1}/{num_epochs}"):
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        output, hidden = model(batch_inputs, hidden)

        hidden = hidden.detach()

        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))  # Flatten the outputs and targets for CrossEntropyLoss
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        with torch.no_grad():
            # Calculate accuracy
            _, predicted_indices = torch.max(output, dim=2)  # Predicted characters

            correct_predictions += (predicted_indices == batch_targets).sum().item()
            total_predictions += batch_targets.size(0) * batch_targets.size(1)  # Total items in this batch

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    accuracy = correct_predictions / total_predictions * 100  # Convert to percentage
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%")

Epoch 1/1:   0%|          | 0/10948 [00:00<?, ?it/s]/var/folders/x1/hpm4ppy95wq62k80d6t196t80000gn/T/ipykernel_1035/1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
/var/folders/x1/hpm4ppy95wq62k80d6t196t80000gn/T/ipykernel_1035/1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)
Epoch 1/1: 100%|██████████| 10948/10948 [21:52<00:00,  8.34it/s]  

Epoch [1/1], Loss: 1.6312, Accuracy: 52.13%


## Check your loss

The training loss of your model when trained with a simple sequence like `"abcdefghijklmnopqrstuvwxyz" * 100` should be extremely close to zero. If that's not the case, go back and fix your bugs ;)

If you have acheived a training loss of 0 or extremley close to 0, then congratulations, lets move on to train your model with a bit more complicated sequence. That is our old favorite book, `warandpeace.txt`.

### Read the `warandpeace.txt` file

In [13]:
# This is Cell #14

sequence = read_file('warandpeace.txt')

### Now Follow the instructions

1. Re-run Cell #5 to re-create character mappings for `warandpeace.txt`
2. Re-run Cell #7 to re-initialize hyperparameters
3. Re-run Cell #8 to split and create training and testing data with `warandpeace.txt` as your corpus
4. Re-run Cell #9 to set up data loaders with `warandpeace.txt` data
5. Re-run Cell #12 to re-initialize a new model object (maybe ask yourself why can't you use the previous model that was trained on the simple `"abc..."` corpus)
6. Re-run Cell #13 to train the new model with `warandpeace.txt` data.
   

## Evaluating the Model

After training, we evaluate the model on the test data.

In [14]:
# Cell #15

with torch.no_grad():
    total_loss, correct_predictions, total_predictions = 0, 0, 0

    hidden = model.init_hidden(batch_size)

    for batch_idx, (batch_inputs, batch_targets) in tqdm(enumerate(test_loader), total=len(test_loader), desc="Testing"):
        batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

        output, hidden = model(batch_inputs, hidden)

        hidden = hidden.detach()

        loss = criterion(output.view(-1, output_size), batch_targets.view(-1))  # Flatten the outputs and targets for CrossEntropyLoss

        # Calculate accuracy
        _, predicted_indices = torch.max(output, dim=2)  # Predicted characters

        correct_predictions += (predicted_indices == batch_targets).sum().item()
        total_predictions += batch_targets.size(0) * batch_targets.size(1)  # Total items in this batch

        total_loss += loss.item()

    avg_loss = total_loss / len(test_loader)
    accuracy = correct_predictions / total_predictions * 100  # Convert to percentage
    print(f"Test Loss: {avg_loss:.4f}, Test Accuracy: {accuracy:.2f}%")

Testing:   0%|          | 0/1216 [00:00<?, ?it/s]/var/folders/x1/hpm4ppy95wq62k80d6t196t80000gn/T/ipykernel_1035/1273917643.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
/var/folders/x1/hpm4ppy95wq62k80d6t196t80000gn/T/ipykernel_1035/1273917643.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(self.targets[idx], dtype=torch.long)
Testing: 100%|██████████| 1216/1216 [00:35<00:00, 34.24it/s]


Test Loss: 1.6610, Test Accuracy: 51.44%


## Generating Text with the Trained Model

In this part of the assignment, your task is to implement the `generate_text` function, which uses a trained RNN model to generate text character-by-character, continuing from a given input. The function will produce an extended sequence by repeatedly predicting and appending the next character to the input.

### What the function is supposed to do?

1. Take an initial input text of length `n` from the user, convert it into indices using a predefined vocabulary (char_to_idx).
2. Use a trained model to predict the next character in the sequence.
3. Append the predicted character to the input, extend the input sequence, and repeat the process until `k` additional characters are generated.
4. Return the generated text, including the original input and the newly predicted characters.


In [16]:
# This is Cell #16

def sample_from_output(logits, temperature=1.0):
    """
    Sample from the logits with temperature scaling.
    logits: Tensor of shape [batch_size, vocab_size] (raw scores, before softmax)
    temperature: a float controlling the randomness (higher = more random)
    """
    # Apply temperature scaling to logits (increase randomness with higher values)
    scaled_logits = logits / temperature  # Scale the logits by temperature
    # Apply softmax to convert logits to probabilities
    probabilities = F.softmax(scaled_logits, dim=1)
    
    # Sample from the probability distribution
    sampled_idx = torch.multinomial(probabilities, 1)  # Sample one index from the probability distribution
    return sampled_idx

def generate_text(model, start_text, n, k, temperature=1.0):
    """
        model: The trained RNN model used for character prediction.
        start_text: The initial string of length `n` provided by the user to start the generation.
        n: The length of the initial input sequence.
        k: The number of additional characters to generate.
        temperature: Optional
        A scaling factor for randomness in predictions. Higher values (e.g., >1) make 
            predictions more random, while lower values (e.g., <1) make predictions more deterministic.
            Default is 1.0.
    """
    start_text = start_text.lower()
    #TODO: Implement the rest of the generate_text function
    # Convert the start text to tensor indices
    input_text = torch.tensor([char_to_idx[char] for char in start_text]).unsqueeze(0).to(device)  # Create input tensor
    generated_text = start_text  # Start with the provided text

    hidden = model.init_hidden(1)  # Initialize the hidden state for the model

    # Generate k additional characters
    for _ in range(k):
        output, hidden = model(input_text, hidden)  # Forward pass through the model
        logits = output[:, -1, :]  # Get the output for the last time step
        sampled_idx = sample_from_output(logits, temperature)  # Sample a new character index
        next_char = idx_to_char[sampled_idx.item()]
        # Append the sampled character to the generated text
        generated_text += next_char
        
        # Update input_text for the next character prediction
        input_text = sampled_idx.view(1,1).to(device) # Append the new character index

    return generated_text

print("Training complete. Now you can generate text.")
while True:
    start_text = input("Enter the initial text (n characters, or 'exit' to quit): ")
    
    if start_text.lower() == 'exit':
        print("Exiting...")
        break
    
    n = len(start_text) 
    k = int(input("Enter the number of characters to generate: "))
    temperature_input = input("Enter the temperature value (1.0 is default, >1 is more random): ")
    temperature = float(temperature_input) if temperature_input else 1.0
    
    completed_text = generate_text(model, start_text, n, k, temperature)
    
    print(f"Generated text: {completed_text}")

Training complete. Now you can generate text.
Generated text: the have the count with the sound the count was the count and the count and the strength and the same the countess with the count and the stood the same the count with the count with the continued the count of the still the french and the prince andrew with the standing the count with the strength of the battle with the continued the standing the continued the continued the same the same the same the french and the standing the same the count with the enemy the same the continued the continued the 
Generated text: the had been love of the other gone deeal went and the beatthe soul, and he of the room so received the pervice and before the since the conversation and the army, and the granders of the candless prince andrew with revence of the be that its face. the first the enemy not and among the deep he had the took ofthe moment the count count of the for the mother for the little distants and the countess whole so the actio

ValueError: invalid literal for int() with base 10: ''

## Report section

In your report, describe your experiments and observations when training the model with two datasets: (1) the sequence `"abcdefghijklmnopqrstuvwxyz" * 100` and (2) the text from `warandpeace.txt`. Include the final loss values for both datasets and discuss how the generated text differed between the two. Explain the impact of changing the `temperature` parameter on the text generation, and provide examples. Reflect on the challenges you faced, your thought process during implementation, and the key insights you gained about RNNs and sequence modeling.


Answer:

When working with the abc sequence, there was a lot more flexibility with adjusting the hyperparameters because each run did not take over 15-20 min. I played around with the hyperparameter values and decided that the best course of action was to have a lot of epochs to the point that the model doesn't start to overtrain. I was able to set the stride and learning rate to a low number to get better accuracy, and as for the rest of the values I mostly stuck to the defaults given to us. This led me to the following Loss and Accuracy for the last epoch and for the test set:- 
Epoch [70/70], Loss: 0.0013, Accuracy: 100.00%
Test Loss: 0.0009, Test Accuracy: 100%

On the other hand, when working with 'War and Peace', the experience was quite different since it is a much longer corpus. Given my hardware, I could no longer rely on a large number of epochs, in fact, epochs above 1 were taking way too long for me to be able to experiment with the other hyperparameters (running times of almost an hour). This was the case with quite a few of the parameters actually. I had to reduce the stride as well. So I kept tweaking and understanding the hyperparameters and for my first 4-6 runs I was not able to exceed 35%. A big part of the reason of then reaching the late 40s and early 50s percentages were the hidden_layers and learning_rate. I found that having hidden_layers around the 100 mark was critical, which makes sense because the model needs to remember more of what its read. Moreover, I decreased the learning_rate to 0.007, which did not hugely affect time complexity and improved the accuracy. The following are the Loss and Accuracy for the 1 epoch and for the test set:-
Epoch [1/1], Loss: 1.6312, Accuracy: 52.13%
Test Loss: 1.6610, Test Accuracy: 51.44%

The generated text for both texts differed based on the 'temperature'. It is important to consider that my input for the initial sequence was always 'the'. First let's talk about the 'abc...' corpus. It mostly ended up continuing the alphabet from the end of the string 'the'. For instance, an output I had for when I used temperature 1 was "thefghijklmnopqrstuvwxyzabcdesfghijklmnopqrstuvwxyzabscefghijklm". This means it is generating based off of strings that exist exactly in the same way in the original corpus, which makes sense because a lower value generates less random text. So when I increased thee 'temperature' to 2.5 for more randomness, I got the text "thefghiroplmnopustowxyzabcdejkopmnyuopqrstuvfyzabghlpiop" as the output. There is more randomness here now. There are some sequences that follow the order of the alphabet, but most of it does not. 

For War and Peace, the results were more interesting. The results are in the output box for cell 16 above. The outputs there are for the 'temperature' values 0.1, 0.5, 0.75, 1, in that order. 0.1 was too low as it lacked randomness which led to a lot repetition of words. For instance, here is a sub-string created within the text "the sound the count was the count and the count and the strength and the same the countess with the count". As you can see, the text doesn't make contextual sense, but every word generated is real, which is a good sign. Then for the 'temperature' 0.5, the generation changed quite a lot. Here is a part of it: "the had been love of the other gone deeal went and the beatthe soul, and he of the room so received the pervice and before the since the conversation and the army". There is a lot of promise here. We are able to see some sub-strings that make sense and long words such as conversation, and there is a lot less repetition. Most words are real and accurate but some words like 'deeal' and 'beatthe' are made up. Then I tried temperature 0.5 and got "the charbled his natsha shout up to find of the hallythe carry of this prince andrew... paral in the fooking the dark out were part". As we can see, the sentences are starting to look more real and contextualized in general, but the issue is that a lot of the words in these sentences just don't exist. Finally, I tried temperature 1 and got "the inverits says, saw wouldly to will for hisglatrion the let theirdearancige he agrafove.the emptisicried vagn nessingly. piby to might onlad poe were a dis is the russian cather did in attain his first with german in blood, comevain...". Now almost all the words are complete nonsense, but again, I see the vision of an actual complete sentence or even a part of a story with actual context.


So, if I had to put it in one sentence, what I learned from my experimentation is that a higher 'temperature' generates text with better context, while a lower 'temperature' generates text with better words.

I faced some challenges with this project mainly because I have never worked with PyTorch before. There was quite a bit of a learning curve here, but I was able to get it done because of the code already provided to us already, and with the help of 383GPT and PyTorch documentation. My thought process during implementation was mostly reasoning through the code provided to us and trying to understand what it does, and connecting that to the concepts we have learned in class. For example, when tweaking the hyperparameters, it was important to understand what each one represented so that I could try to predict the outcome of my next run (will my accuracy improve or not?) and also whether changing the hyperparameter by a certain amount would increase the time complexity too much. The key insights I gained about RNNs and sequence modeling were mostly about how critical hyperparameter optimization is. I was able to first hand experience its importance and it helped me think of its larger applications and how optimizing them on a larger scale could be revolutionary to so many types of applications. I also liked the fact that in doing this project I got to read a little about active research in this field and understood that there is still so much to learn in it.
